In [1]:
import pandas as pd

df = pd.read_csv("/content/iron_steel_preprocessed.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/iron_steel_preprocessed.csv'

In [ ]:
df.isnull().sum().sum()

np.int64(0)

In [ ]:

import pandas as pd

df = pd.read_csv("/content/iron-and-steel_emissions_sources_v5_7_0.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(60039, 43)
['source_id', 'source_name', 'source_type', 'iso3_country', 'sector', 'subsector', 'start_time', 'end_time', 'lat', 'lon', 'geometry_ref', 'gas', 'emissions_quantity', 'temporal_granularity', 'activity', 'activity_units', 'emissions_factor', 'emissions_factor_units', 'capacity', 'capacity_units', 'capacity_factor', 'other1', 'other1_def', 'other2', 'other2_def', 'other3', 'other3_def', 'other4', 'other4_def', 'other5', 'other5_def', 'other6', 'other6_def', 'other7', 'other7_def', 'other8', 'other8_def', 'other9', 'other9_def', 'other10', 'other10_def', 'created_date', 'modified_date']


,source_id,source_name,source_type,iso3_country,sector,subsector,start_time,end_time,lat,lon,...,other7,other7_def,other8,other8_def,other9,other9_def,other10,other10_def,created_date,modified_date
0,42740981,Aceria Angola Bengo steel plant,BF/BOF,AGO,manufacturing,iron-and-steel,2021-01-01 00:00:00,2021-01-31 00:00:00,-8.582603,13.405239,...,NaN,percentage_of_scrap_metal_utilized,0.9,Percentage of IPPU emissions,NaN,field_not_included,NaN,field_not_included,2025-08-04 00:00:00,2026-05-07 08:07:09.261742
1,42740981,Aceria Angola Bengo steel plant,BF/BOF,AGO,manufacturing,iron-and-steel,2021-02-01 00:00:00,2021-02-28 00:00:00,-8.582603,13.405239,...,NaN,percentage_of_scrap_metal_utilized,0.9,Percentage of IPPU emissions,NaN,field_not_included,NaN,field_not_included,2025-08-04 00:00:00,2026-05-07 08:07:09.261742
2,42740981,Aceria Angola Bengo steel plant,BF/BOF,AGO,manufacturing,iron-and-steel,2021-03-01 00:00:00,2021-03-31 00:00:00,-8.582603,13.405239,...,NaN,percentage_of_scrap_metal_utilized,0.9,Percentage of IPPU emissions,NaN,field_not_included,NaN,field_not_included,2025-08-04 00:00:00,2026-05-07 08:07:09.261742
3,42740981,Aceria Angola Bengo steel plant,BF/BOF,AGO,manufacturing,iron-and-steel,2021-04-01 00:00:00,2021-04-30 00:00:00,-8.582603,13.405239,...,NaN,percentage_of_scrap_metal_utilized,0.9,Percentage of IPPU emissions,NaN,field_not_included,NaN,field_not_included,2025-08-04 00:00:00,2026-05-07 08:07:09.261742
4,42740981,Aceria Angola Bengo steel plant,BF/BOF,AGO,manufacturing,iron-and-steel,2021-05-01 00:00:00,2021-05-31 00:00:00,-8.582603,13.405239,...,NaN,percentage_of_scrap_metal_utilized,0.9,Percentage of IPPU emissions,NaN,field_not_included,NaN,field_not_included,2025-08-04 00:00:00,2026-05-07 08:07:09.261742


In [ ]:
df.columns.tolist()

['source_id',
 'start_time',
 'iso3_country',
 'source_type',
 'sector',
 'subsector',
 'gas',
 'activity',
 'activity_units',
 'emissions_factor',
 'emissions_factor_units',
 'capacity',
 'capacity_units',
 'capacity_factor',
 'lat',
 'lon',
 'year',
 'month',
 'quarter',
 'emission_lag_1',
 'emission_lag_3',
 'emission_lag_6',
 'emission_lag_12',
 'rolling_mean_3',
 'rolling_mean_6',
 'rolling_mean_12',
 'emissions_quantity']

In [ ]:
df.isnull().sum().sum()

np.int64(0)

In [ ]:
!pip install -q tensorflow scikit-learn
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout,
    Bidirectional,
    BatchNormalization
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)
from tensorflow.keras.losses import Huber

# ==========================
# LOAD DATA
# ==========================

df = pd.read_csv("/content/iron_steel_preprocessed.csv")

# ==========================
# DROP NON-PREDICTIVE
# ==========================

drop_cols = [
    "source_id",
    "start_time"
]

df = df.drop(columns=drop_cols)

# ==========================
# TARGET
# ==========================

TARGET = "emissions_quantity"

X = df.drop(columns=[TARGET])
y = df[TARGET]

# ==========================
# TRAIN TEST SPLIT
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================
# SCALING
# ==========================

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================
# RESHAPE FOR LSTM
# ==========================

X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

# ==========================
# MODEL
# ==========================

model = Sequential([

    Bidirectional(
        LSTM(
            128,
            return_sequences=True
        ),
        input_shape=(
            X_train.shape[1],
            1
        )
    ),

    BatchNormalization(),

    Dropout(0.3),

    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        )
    ),

    BatchNormalization(),

    Dropout(0.2),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)

])

# ==========================
# COMPILE
# ==========================

optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001,
    clipnorm=1.0
)

model.compile(
    optimizer=optimizer,
    loss=Huber(),
    metrics=["mae"]
)

model.summary()

# ==========================
# CALLBACKS
# ==========================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_bilstm.keras",
    monitor="val_loss",
    save_best_only=True
)

# ==========================
# TRAIN
# ==========================

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=256,
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint
    ],
    verbose=1
)

# ==========================
# PREDICT
# ==========================

preds = model.predict(X_test).flatten()

# ==========================
# METRICS
# ==========================

mae = mean_absolute_error(
    y_test,
    preds
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        preds
    )
)

r2 = r2_score(
    y_test,
    preds
)

mape = np.mean(
    np.abs(
        (y_test - preds) /
        np.maximum(
            np.abs(y_test),
            1e-8
        )
    )
) * 100

print("\n======================")
print("RESULTS")
print("======================")

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

# ==========================
# SAMPLE PREDICTIONS
# ==========================

comparison = pd.DataFrame({
    "Actual": y_test.values[:20],
    "Predicted": preds[:20]
})

print("\nSample Predictions")
print(comparison)

# ==========================
# SAVE SCALER
# ==========================

import joblib

joblib.dump(
    scaler,
    "scaler.pkl"
)

print("\nModel Saved: best_bilstm.keras")
print("Scaler Saved: scaler.pkl")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       133,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 24, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 309,377 (1.18 MB)

 Trainable params: 308,609 (1.18 MB)

 Non-trainable params: 768 (3.00 KB)

Epoch 1/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - loss: 1.1701 - mae: 1.5548 - val_loss: 6.4225 - val_mae: 6.9225 - learning_rate: 0.0010
Epoch 2/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.4124 - mae: 0.7587 - val_loss: 3.9877 - val_mae: 4.4875 - learning_rate: 0.0010
Epoch 3/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 0.3618 - mae: 0.6970 - val_loss: 2.2058 - val_mae: 2.6792 - learning_rate: 0.0010
Epoch 4/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.3428 - mae: 0.6719 - val_loss: 0.6443 - val_mae: 1.0952 - learning_rate: 0.0010
Epoch 5/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - loss: 0.3045 - mae: 0.6252 - val_loss: 0.1930 - val_mae: 0.4684 - learning_rate: 0.0010
Epoch 6/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 0.2832 - mae: 0.5993 - val_loss: 0.1896 - val_mae: 0.4436 - learning_rate: 0.0010
Epoch 7/100
151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 0.2700 - mae: 0.5812 - val_loss: 0.6727 - val_mae: 1.0692 - learning_rate

In [ ]:
from sklearn.metrics import r2_score

print("R2 =", r2_score(y_test, preds))

R2 = 0.952268629973343
